# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第3章 研究倫理とオープンサイエンス ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch03_p_hacking_simulation.R`

対象章: 第3章「研究倫理とオープンサイエンス」

「HARKingとp値ハッキング」節の作例。Simmons, Nelson, and Simonsohn (2011)
が指摘した「開示されない柔軟性」が、真に効果が存在しない場合でも
偽陽性率を名目上の5%から大きく引き上げてしまうことを、架空のデータへの
実際のシミュレーションで確認する。実測データではない。

設計: 真に効果がない(2群の母平均が同じ)状況で、以下の2つの
「柔軟性」を研究者が使えるとする。
  (a) 任意打ち切り: n=20で有意でなければ、n=40まで追加してから再検定する
  (b) 従属変数の選択: 相関のある2つの従属変数を測定し、p値の小さい方を報告する
これらを組み合わせた場合の偽陽性率を、単純な1回きりの検定と比較する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)
n_sim <- 5000

simulate_naive <- function() {
  # 単純な1回きりの検定(n=20、DV1本のみ)。真の効果はゼロ。
  a <- rnorm(20, 0, 1); b <- rnorm(20, 0, 1)
  t.test(a, b)$p.value
}

simulate_flexible <- function() {
  # (a) 任意打ち切り + (b) 従属変数の選択、を両方使う場合
  a1 <- rnorm(20, 0, 1); b1 <- rnorm(20, 0, 1)
  # DV2はDV1と相関r=0.5を持つよう生成(同じ参加者の別指標という想定)
  a2 <- 0.5 * a1 + rnorm(20, 0, sqrt(1 - 0.5^2))
  b2 <- 0.5 * b1 + rnorm(20, 0, sqrt(1 - 0.5^2))
  p1 <- t.test(a1, b1)$p.value
  p2 <- t.test(a2, b2)$p.value
  p_best_n20 <- min(p1, p2)
  if (p_best_n20 < .05) return(p_best_n20)

  # 有意でなければ、さらに20名追加してn=40で再検定
  a1b <- c(a1, rnorm(20, 0, 1)); b1b <- c(b1, rnorm(20, 0, 1))
  a2b <- c(a2, 0.5 * a1b[21:40] + rnorm(20, 0, sqrt(1 - 0.5^2)))
  b2b <- c(b2, 0.5 * b1b[21:40] + rnorm(20, 0, sqrt(1 - 0.5^2)))
  p1b <- t.test(a1b, b1b)$p.value
  p2b <- t.test(a2b, b2b)$p.value
  min(p1b, p2b)
}

p_naive    <- replicate(n_sim, simulate_naive())
p_flexible <- replicate(n_sim, simulate_flexible())

fpr_naive    <- mean(p_naive < .05)
fpr_flexible <- mean(p_flexible < .05)

cat("単純な1回きりの検定での偽陽性率:", round(fpr_naive * 100, 1), "%\n")
cat("2種の柔軟性を組み合わせた場合の偽陽性率:", round(fpr_flexible * 100, 1), "%\n")
cat("名目上の水準(5%)に対する倍率:", round(fpr_flexible / .05, 2), "倍\n")
